# Hybrid Quantum-Classical Model for Exoplanet Biosignature Detection

**Dataset**: ADC2023 (Ariel Data Challenge) — 41,423 exoplanet transmission spectra, 52 wavelength bins

**Architecture**: Dual-encoder (aux FFN + spectral Conv1d) → Fusion with LayerNorm → Variational Quantum Circuit (12 qubits) → Prediction head with skip connections

**Targets**: log_H2O, log_CO2, log_CO, log_CH4, log_NH3 (log10 volume mixing ratios)

## 1. Configuration

In [1]:
import json
import math
import os
import random
import time
from contextlib import nullcontext
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, Optional

os.environ.setdefault("XDG_CACHE_HOME", str(Path(".cache").resolve()))
os.environ.setdefault("MPLCONFIGDIR", str(Path(".matplotlib-cache").resolve()))

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pennylane as qml
import torch
import torch.nn as nn
from IPython.display import display
from sklearn.model_selection import train_test_split

try:
    from tqdm.auto import tqdm
except ImportError:
    class _TqdmFallback:
        def __init__(self, iterable=None, total=None, desc=None, leave=True):
            self.iterable = iterable

        def __iter__(self):
            return iter(self.iterable)

        def set_postfix(self, **kwargs):
            pass

        @staticmethod
        def write(message):
            print(message)

    def tqdm(iterable=None, total=None, desc=None, leave=True):
        return _TqdmFallback(iterable=iterable, total=total, desc=desc, leave=leave)

# --- Constants ---
AUX_FEATURE_COLS = [
    "star_mass_kg", "star_radius_m", "star_temperature",
    "planet_mass_kg", "planet_orbital_period", "planet_distance",
    "planet_surface_gravity", "log10_noise_mean",
]
TARGET_COLS = ["log_H2O", "log_CO2", "log_CO", "log_CH4", "log_NH3"]

# --- Hyperparameters ---
SEED = 42
DATA_ROOT_CANDIDATES = [
    Path("FullDataset"),
    Path("ariel-ml-dataset"),
    Path("../FullDataset"),
    Path("../ariel-ml-dataset"),
]
OUTPUT_DIR = Path("outputs/hybrid_quantum_biosignature")
TRAIN_BATCH_SIZE = 256
EVAL_BATCH_SIZE = 8192
MAX_EPOCHS = 30
EARLY_STOP_PATIENCE = 6
SCHEDULER_PATIENCE = 5
CLASSICAL_LR = 2e-3
QUANTUM_LR = 6e-4
WEIGHT_DECAY = 1e-4
GRADIENT_CLIP = 5.0
DROPOUT = 0.05
VAL_FRACTION = 0.10

# Model dimensions
AUX_HIDDEN = 64
AUX_OUT = 32
SPECTRAL_HIDDEN = 64
SPECTRAL_OUT = 32
FUSION_HIDDEN = 48
HEAD_HIDDEN = 96
QNN_QUBITS = 12
QNN_DEPTH = 2


def resolve_data_root(candidates):
    for root in candidates:
        train_dir = root / "TrainingData"
        if (train_dir / "AuxillaryTable.csv").exists() and (train_dir / "SpectralData.hdf5").exists():
            return root
    searched = "\n".join(f" - {path.resolve()}" for path in candidates)
    raise FileNotFoundError(
        "Could not locate the Ariel dataset. Put the extracted dataset in one of:\n"
        f"{searched}"
    )


def resolve_quantum_device():
    for name in ("lightning.qubit", "default.qubit"):
        try:
            dev_kwargs = {"wires": 1}
            if name.startswith("lightning."):
                dev_kwargs["c_dtype"] = np.complex64
            qml.device(name, **dev_kwargs)
            return name
        except Exception:
            continue
    raise RuntimeError("No supported PennyLane device found. Install pennylane-lightning or use default.qubit.")


DATA_ROOT = resolve_data_root(DATA_ROOT_CANDIDATES)
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
QUANTUM_DEVICE = resolve_quantum_device()

print(f"PyTorch: {torch.__version__}")
print(f"PennyLane: {qml.__version__}")
print(f"Data root: {DATA_ROOT.resolve()}")
print(f"Device: {DEVICE} | Quantum: {QUANTUM_DEVICE}")
print(f"Qubits: {QNN_QUBITS} | Depth: {QNN_DEPTH}")

PyTorch: 2.10.0
PennyLane: 0.44.1
Data root: /Users/jkw/Documents/uni/axion/hack4sages/FullDataset
Device: mps | Quantum: lightning.qubit
Qubits: 12 | Depth: 2


/Users/jkw/Documents/uni/axion/hack4sages/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

cpu_threads = max(1, min(os.cpu_count() or 1, 32))
torch.set_num_threads(cpu_threads)

# Jupyter may have already initialized interop threads in this kernel.
try:
    torch.set_num_interop_threads(max(1, min(cpu_threads // 2, 8)))
except RuntimeError as exc:
    print(f"Skipping torch.set_num_interop_threads: {exc}")

## 2. Data Loading & Preprocessing

Load from ADC2023 format: `AuxillaryTable.csv` (stellar/planetary properties), `FM_Parameter_Table.csv` (ground truth VMRs), `SpectralData.hdf5` (per-planet transmission spectra with noise).

In [3]:
train_dir = DATA_ROOT / "TrainingData"
aux_path = train_dir / "AuxillaryTable.csv"
gt_path = train_dir / "Ground Truth Package" / "FM_Parameter_Table.csv"
spectral_path = train_dir / "SpectralData.hdf5"

# Load auxiliary features
aux_df = pd.read_csv(aux_path)
gt_df = pd.read_csv(gt_path)
labels = aux_df.merge(gt_df[["planet_ID"] + TARGET_COLS], on="planet_ID", how="inner").reset_index(drop=True)

n_samples = len(labels)
print(f"Samples: {n_samples}")
print(f"Aux features: {AUX_FEATURE_COLS[:7]}")
print(f"Targets: {TARGET_COLS}")
print(f"Using spectra file: {spectral_path.resolve()}")

# Load spectra from HDF5 (per-planet groups)
with h5py.File(spectral_path, "r") as h:
    first_key = list(h.keys())[0]
    wavelength_um = np.asarray(h[first_key]["instrument_wlgrid"][:], dtype=np.float32)
    n_bins = len(wavelength_um)
    noisy_spectra = np.empty((n_samples, n_bins), dtype=np.float32)
    noise_arr = np.empty((n_samples, n_bins), dtype=np.float32)
    for i, pid in enumerate(labels["planet_ID"].values):
        grp = h[f"Planet_{pid}"]
        noisy_spectra[i] = grp["instrument_spectrum"][:]
        noise_arr[i] = grp["instrument_noise"][:]

# Derive log10 noise mean as 8th aux feature
labels["log10_noise_mean"] = np.log10(np.clip(noise_arr.mean(axis=1), 1e-10, None))

print(f"Spectra shape: {noisy_spectra.shape}")
print(f"Wavelength range: {wavelength_um.min():.2f} - {wavelength_um.max():.2f} um")

Samples: 41423
Aux features: ['star_mass_kg', 'star_radius_m', 'star_temperature', 'planet_mass_kg', 'planet_orbital_period', 'planet_distance', 'planet_surface_gravity']
Targets: ['log_H2O', 'log_CO2', 'log_CO', 'log_CH4', 'log_NH3']
Using spectra file: /Users/jkw/Documents/uni/axion/hack4sages/FullDataset/TrainingData/SpectralData.hdf5
Spectra shape: (41423, 52)
Wavelength range: 0.55 - 7.28 um


In [4]:
# Build feature arrays
aux_raw = labels[AUX_FEATURE_COLS].to_numpy(dtype=np.float32)
targets_raw = labels[TARGET_COLS].to_numpy(dtype=np.float32)

# Per-sample spectral normalization (removes Rp/Rs baseline)
per_sample_mean = noisy_spectra.mean(axis=1, keepdims=True)
per_sample_mean = np.where(per_sample_mean == 0, 1.0, per_sample_mean)
spectra_raw = (noisy_spectra / per_sample_mean)[:, None, :].astype(np.float32)  # (N, 1, 52)

print(f"Aux array: {aux_raw.shape}")
print(f"Spectra array: {spectra_raw.shape}")
print(f"Targets array: {targets_raw.shape}")

Aux array: (41423, 8)
Spectra array: (41423, 1, 52)
Targets array: (41423, 5)


### Train / Validation / Test Split

90% train+val / 10% test, then 90/10 within train+val. Scalers fit on train only — no data leakage.

In [5]:
# Split: 81% train, 9% val, 10% test
all_idx = np.arange(n_samples)
train_val_idx, test_idx = train_test_split(all_idx, test_size=0.10, random_state=SEED, shuffle=True)
train_idx, val_idx = train_test_split(train_val_idx, test_size=VAL_FRACTION, random_state=SEED, shuffle=True)
train_idx, val_idx, test_idx = np.sort(train_idx), np.sort(val_idx), np.sort(test_idx)

# Verify no overlap
assert len(set(train_idx) & set(val_idx)) == 0
assert len(set(train_idx) & set(test_idx)) == 0
assert len(set(val_idx) & set(test_idx)) == 0
assert len(train_idx) + len(val_idx) + len(test_idx) == n_samples

print(f"Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)} | Total: {n_samples}")

Train: 33552 | Val: 3728 | Test: 4143 | Total: 41423


In [6]:
# Fit scalers on TRAIN SET ONLY
def fit_scaler(arr):
    mean = arr.astype(np.float64).mean(axis=0).astype(np.float32)
    scale = arr.astype(np.float64).std(axis=0).astype(np.float32)
    scale = np.where(scale == 0, 1.0, scale)
    return mean, scale

aux_mean, aux_scale = fit_scaler(aux_raw[train_idx])
target_mean, target_scale = fit_scaler(targets_raw[train_idx])
spec_mean, spec_scale = fit_scaler(spectra_raw[train_idx, 0, :])

def scale_aux(a):    return ((a - aux_mean) / aux_scale).astype(np.float32)
def scale_target(t): return ((t - target_mean) / target_scale).astype(np.float32)
def scale_spec(s):   return ((s - spec_mean[None, None, :]) / spec_scale[None, None, :]).astype(np.float32)
def unscale_target(t): return (t * target_scale + target_mean).astype(np.float32)

# Build tensors per split
def make_tensors(idx):
    return (
        torch.from_numpy(scale_aux(aux_raw[idx])),
        torch.from_numpy(scale_spec(spectra_raw[idx])),
        torch.from_numpy(scale_target(targets_raw[idx])),
        targets_raw[idx].copy(),
    )

train_aux, train_spec, train_tgt, train_raw_tgt = make_tensors(train_idx)
val_aux, val_spec, val_tgt, val_raw_tgt = make_tensors(val_idx)
test_aux, test_spec, test_tgt, test_raw_tgt = make_tensors(test_idx)

print(f"Train tensors: aux={train_aux.shape}, spec={train_spec.shape}, tgt={train_tgt.shape}")

Train tensors: aux=torch.Size([33552, 8]), spec=torch.Size([33552, 1, 52]), tgt=torch.Size([33552, 5])


## 3. Model Architecture

```
Aux (8)  ──→ AuxEncoder (FFN: 8→64→64→32) ──────────────────────┐
                                              ↓                    │ skip
Spectra (52) → SpectralEncoder (Conv1d→pool→32) ──────────────┐  │
                                              ↓                 │  │
                    FusionEncoder (cat→FFN→LayerNorm→tanh·π→12)  │  │
                                              ↓                 │  │
                    QuantumCircuit (12 qubits, depth 2, VQC)    │  │
                                              ↓                 │  │
                    Concatenate [quantum(12) + fusion(12) + spectral(32) + aux(32)] = 88
                                              ↓
                    PredictionHead (FFN: 88→96→96→5) + residual from quantum
                                              ↓
                    Output: 5 log VMR predictions
```

In [7]:
class AuxEncoder(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, out_dim), nn.GELU(),
        )

    def forward(self, x):
        return self.net(x)

In [8]:
class SpectralEncoder(nn.Module):
    def __init__(self, in_channels, hidden_dim, out_dim, dropout):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(in_channels, 32, kernel_size=7, padding=3), nn.GELU(),
            nn.Conv1d(32, hidden_dim, kernel_size=5, stride=2, padding=2), nn.GELU(),
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=3, padding=1), nn.GELU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(hidden_dim, out_dim), nn.GELU(), nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.proj(self.conv(x))

In [9]:
class FusionEncoder(nn.Module):
    def __init__(self, aux_dim, spec_dim, hidden_dim, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(aux_dim + spec_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, out_dim), nn.LayerNorm(out_dim),
        )

    def forward(self, aux_feat, spectral_feat):
        return torch.tanh(self.net(torch.cat([aux_feat, spectral_feat], dim=-1))) * math.pi

In [10]:
class QuantumBlock(nn.Module):
    def __init__(self, n_qubits, depth, quantum_device_name):
        super().__init__()
        self.n_qubits = n_qubits
        self.num_blocks = depth // 2
        self.num_weights = 3 * n_qubits * self.num_blocks
        self.weights = nn.Parameter(0.5 * torch.randn(self.num_weights, dtype=torch.float32))

        dev_kwargs = {"wires": n_qubits}
        if quantum_device_name.startswith("lightning."):
            dev_kwargs["c_dtype"] = np.complex64
        dev = qml.device(quantum_device_name, **dev_kwargs)

        @qml.qnode(dev, interface="torch", diff_method="adjoint")
        def circuit(inputs, weights):
            for q in range(n_qubits):
                qml.RY(inputs[..., q], wires=q)
            idx = 0
            for _ in range(self.num_blocks):
                for q in range(n_qubits):
                    qml.RY(weights[idx], wires=q); idx += 1
                for q in range(n_qubits):
                    qml.CNOT(wires=[q, (q + 1) % n_qubits])
                for q in range(n_qubits):
                    qml.RZ(weights[idx], wires=q); idx += 1
                for q in range(n_qubits):
                    qml.CRX(weights[idx], wires=[q, (q + 1) % n_qubits]); idx += 1
            return [qml.expval(qml.PauliZ(q)) for q in range(n_qubits)]

        self.qnode = circuit

    def forward(self, x):
        out = self.qnode(x.float(), self.weights)
        if isinstance(out, (list, tuple)):
            out = torch.stack(tuple(out), dim=-1)
        return out.float().to(x.device)

print(f"Quantum circuit: {QNN_QUBITS} qubits, depth {QNN_DEPTH}, {3 * QNN_QUBITS * (QNN_DEPTH // 2)} trainable params")

Quantum circuit: 12 qubits, depth 2, 36 trainable params


In [11]:
class AtmosphereHead(nn.Module):
    def __init__(self, in_dim, latent_dim, hidden_dim, n_targets, dropout):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, n_targets),
        )
        self.residual = nn.Linear(latent_dim, n_targets)

    def forward(self, head_in, quantum_feat):
        return self.mlp(head_in) + self.residual(quantum_feat)

In [12]:
class HybridQuantumModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.aux_enc = AuxEncoder(len(AUX_FEATURE_COLS), AUX_HIDDEN, AUX_OUT, DROPOUT)
        self.spec_enc = SpectralEncoder(1, SPECTRAL_HIDDEN, SPECTRAL_OUT, DROPOUT)
        self.fusion = FusionEncoder(AUX_OUT, SPECTRAL_OUT, FUSION_HIDDEN, QNN_QUBITS)
        self.quantum = QuantumBlock(QNN_QUBITS, QNN_DEPTH, QUANTUM_DEVICE)
        head_in_dim = QNN_QUBITS * 2 + AUX_OUT + SPECTRAL_OUT  # 88
        self.head = AtmosphereHead(head_in_dim, QNN_QUBITS, HEAD_HIDDEN, len(TARGET_COLS), DROPOUT)

    def forward(self, aux, spectra):
        aux_feat = self.aux_enc(aux)           # (B, 8) -> (B, 32)
        spec_feat = self.spec_enc(spectra)      # (B, 1, 52) -> (B, 32)
        latent = self.fusion(aux_feat, spec_feat)  # (B, 64) -> (B, 12)
        quantum_feat = self.quantum(latent)     # (B, 12) -> (B, 12) via VQC
        head_in = torch.cat([quantum_feat, latent, aux_feat, spec_feat], dim=-1)  # (B, 88)
        return self.head(head_in, quantum_feat) # (B, 88) -> (B, 5)

    def classical_parameters(self):
        for m in (self.aux_enc, self.spec_enc, self.fusion, self.head):
            yield from m.parameters()

    def quantum_parameters(self):
        yield from self.quantum.parameters()

model = HybridQuantumModel().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
n_classical = sum(p.numel() for p in model.classical_parameters())
n_quantum = sum(p.numel() for p in model.quantum_parameters())
print(f"Total parameters: {n_params:,} (classical: {n_classical:,}, quantum: {n_quantum:,})")

Total parameters: 53,982 (classical: 53,946, quantum: 36)


## 4. Training Loop

- AdamW with separate LRs: classical (2e-3) vs quantum (6e-4)
- ReduceLROnPlateau on validation loss
- Early stopping (patience=6)
- Gradient clipping: 5.0 (classical), 1.0 (quantum)
- Checkpoint saved on every val improvement

In [13]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

classical_params = list(model.classical_parameters())
quantum_params = list(model.quantum_parameters())

optimizer = torch.optim.AdamW([
    {"params": classical_params, "lr": CLASSICAL_LR, "weight_decay": WEIGHT_DECAY},
    {"params": quantum_params,   "lr": QUANTUM_LR,   "weight_decay": 0.0},
])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=SCHEDULER_PATIENCE,
)
loss_fn = nn.MSELoss()


def evaluate(aux, spec, tgt, raw_tgt):
    model.eval()
    aux = aux.to(DEVICE)
    spec = spec.to(DEVICE)
    tgt = tgt.to(DEVICE)
    with torch.inference_mode():
        pred = model(aux, spec)
        loss = loss_fn(pred, tgt).item()
    pred_orig = unscale_target(pred.detach().cpu().numpy())
    rmse = np.sqrt(np.mean((pred_orig - raw_tgt) ** 2, axis=0))
    return loss, rmse, rmse.mean(), pred_orig


history = []
best_val_loss = float("inf")
best_epoch = -1
best_state = None
patience_left = EARLY_STOP_PATIENCE
n_batches = math.ceil(len(train_aux) / TRAIN_BATCH_SIZE)

print(f"Training: {len(train_aux)} samples, {n_batches} batches/epoch, max {MAX_EPOCHS} epochs")
print(f"{'='*90}")

for epoch in range(MAX_EPOCHS):
    model.train()
    t0 = time.perf_counter()
    perm = torch.randperm(len(train_aux), generator=torch.Generator().manual_seed(SEED + epoch))
    batch_losses = []
    batch_starts = range(0, len(train_aux), TRAIN_BATCH_SIZE)
    progress = tqdm(batch_starts, total=n_batches, desc=f"Epoch {epoch + 1}/{MAX_EPOCHS}", leave=False)

    for batch_num, b in enumerate(progress, start=1):
        idx = perm[b:b + TRAIN_BATCH_SIZE]
        batch_aux = train_aux[idx].to(DEVICE)
        batch_spec = train_spec[idx].to(DEVICE)
        batch_tgt = train_tgt[idx].to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        pred = model(batch_aux, batch_spec)
        loss = loss_fn(pred, batch_tgt)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(classical_params, GRADIENT_CLIP)
        torch.nn.utils.clip_grad_norm_(quantum_params, 1.0)
        optimizer.step()

        loss_value = loss.item()
        batch_losses.append(loss_value)
        progress.set_postfix(loss=f"{loss_value:.5f}")

        if batch_num % 25 == 0 or batch_num == n_batches:
            tqdm.write(
                f"Epoch {epoch + 1:2d} | Batch {batch_num:3d}/{n_batches} | loss={loss_value:.5f}"
            )

    train_loss = np.mean(batch_losses)
    val_loss, val_rmse, val_rmse_mean, _ = evaluate(val_aux, val_spec, val_tgt, val_raw_tgt)
    scheduler.step(val_loss)
    elapsed = time.perf_counter() - t0

    row = {
        "epoch": epoch + 1, "train_loss": train_loss, "val_loss": val_loss,
        "val_rmse_mean": val_rmse_mean, "time": elapsed,
        "lr_classical": optimizer.param_groups[0]["lr"],
        "lr_quantum": optimizer.param_groups[1]["lr"],
    }
    for name, r in zip(TARGET_COLS, val_rmse):
        row[f"rmse_{name}"] = float(r)
    history.append(row)

    rmse_str = " | ".join(f"{c}={r:.4f}" for c, r in zip(TARGET_COLS, val_rmse))
    print(
        f"Epoch {epoch+1:2d}/{MAX_EPOCHS} | train={train_loss:.5f} | val={val_loss:.5f} | "
        f"rmse={val_rmse_mean:.4f} | {elapsed:.0f}s | lr=({row['lr_classical']:.1e}, {row['lr_quantum']:.1e})"
    )
    print(f"  Validation RMSE: {rmse_str}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch + 1
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        patience_left = EARLY_STOP_PATIENCE
        torch.save({"epoch": best_epoch, "val_loss": best_val_loss, "model_state_dict": best_state},
                   OUTPUT_DIR / "best_model.pt")
        print(f"  -> Checkpoint saved (epoch {best_epoch}, val_loss={best_val_loss:.5f})")
    else:
        patience_left -= 1
        if patience_left <= 0:
            print(f"Early stopping at epoch {epoch + 1}.")
            break

print(f"{'='*90}")
print(f"Best epoch: {best_epoch} | Best val loss: {best_val_loss:.5f}")
model.load_state_dict(best_state)

Training: 33552 samples, 132 batches/epoch, max 30 epochs


Epoch 1/30:  19%|▏| 25/132 [03:06<13:21,  7.49s/it, loss=0.96948

Epoch  1 | Batch  25/132 | loss=0.96948


Epoch 1/30:  38%|▍| 50/132 [06:11<10:06,  7.39s/it, loss=0.91614

Epoch  1 | Batch  50/132 | loss=0.91614


Epoch 1/30:  57%|▌| 75/132 [09:19<07:11,  7.57s/it, loss=0.83676

Epoch  1 | Batch  75/132 | loss=0.83676


Epoch 1/30:  76%|▊| 100/132 [12:17<03:51,  7.22s/it, loss=0.7896

Epoch  1 | Batch 100/132 | loss=0.78962


KeyboardInterrupt: 

## 5. Evaluation

In [ ]:
test_loss, test_rmse, test_rmse_mean, test_pred = evaluate(test_aux, test_spec, test_tgt, test_raw_tgt)

print(f"Test Loss: {test_loss:.5f}")
print(f"Test RMSE (mean): {test_rmse_mean:.4f}")
print()
results_df = pd.DataFrame({"target": TARGET_COLS, "RMSE": test_rmse})
display(results_df)

In [ ]:
hf = pd.DataFrame(history)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(hf["epoch"], hf["train_loss"], label="Train")
axes[0].plot(hf["epoch"], hf["val_loss"], label="Val")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("MSE Loss")
axes[0].set_title("Loss Curves"); axes[0].legend()

axes[1].plot(hf["epoch"], hf["val_rmse_mean"])
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Mean RMSE")
axes[1].set_title("Validation RMSE")

for col in TARGET_COLS:
    axes[2].plot(hf["epoch"], hf[f"rmse_{col}"], label=col)
axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("RMSE")
axes[2].set_title("Per-Target RMSE"); axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "training_curves.png", dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for i, (col, ax) in enumerate(zip(TARGET_COLS, axes)):
    true = test_raw_tgt[:, i]
    pred = test_pred[:, i]
    ax.scatter(true, pred, alpha=0.15, s=3)
    lims = [min(true.min(), pred.min()), max(true.max(), pred.max())]
    ax.plot(lims, lims, "r--", lw=1)
    ax.set_xlabel("True"); ax.set_ylabel("Predicted")
    ax.set_title(f"{col}\nRMSE={test_rmse[i]:.3f}")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "scatter_predictions.png", dpi=150)
plt.show()

## 6. Save Artifacts

In [ ]:
# Save final model
torch.save({
    "feature_cols": AUX_FEATURE_COLS, "target_cols": TARGET_COLS,
    "best_epoch": best_epoch, "best_val_loss": best_val_loss,
    "model_state_dict": best_state,
    "qnn_qubits": QNN_QUBITS, "qnn_depth": QNN_DEPTH,
}, OUTPUT_DIR / "best_model.pt")

# Save scalers
scalers = {
    "aux": {"mean": aux_mean.tolist(), "scale": aux_scale.tolist()},
    "target": {"mean": target_mean.tolist(), "scale": target_scale.tolist()},
    "spectral": {"mean": spec_mean.tolist(), "scale": spec_scale.tolist()},
}
(OUTPUT_DIR / "scalers.json").write_text(json.dumps(scalers, indent=2))

# Save history
hf.to_csv(OUTPUT_DIR / "history.csv", index=False)

# Save test predictions
pred_df = pd.DataFrame({"planet_ID": labels.iloc[test_idx]["planet_ID"].values})
for i, col in enumerate(TARGET_COLS):
    pred_df[f"true_{col}"] = test_raw_tgt[:, i]
    pred_df[f"pred_{col}"] = test_pred[:, i]
pred_df.to_csv(OUTPUT_DIR / "test_predictions.csv", index=False)

print(f"Artifacts saved to {OUTPUT_DIR}/")
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {f.name} ({f.stat().st_size / 1024:.0f} KB)")